# BÁO CÁO TỔNG KẾT & ĐỐI CHUẨN MÔ HÌNH HỌC MÁY NÂNG CAO (VERSION 3 - TUNED RFM)
## So Sánh, Đánh Giá Toàn Diện & Lựa Chọn Kiến Trúc Tối Ưu Giữa 3 Mô Hình Phân Cụm Tinh Chỉnh (K-Means, Hierarchical, DBSCAN)

**Đề tài:** Phân khúc khách hàng bằng RFM và hành vi mua hàng (Online Retail II)  
**Nhóm thực hiện:** Nhóm 05 - Lớp DCT124C1 - Trường Đại học Sài Gòn (SGU)

Báo cáo này đóng vai trò là **tổng kết đối chuẩn cấp cao nhất (Meta-Evaluation)** cho toàn bộ đề tài: Đặt 3 mô hình phân cụm đã được tinh chỉnh siêu tham số tối ưu (Version 3 - Tuned RFM) lên bàn cân đối chiếu trực diện trên cùng không gian đặc trưng RFM chuẩn hóa và 3 tập dữ liệu độc lập (Train / Validation / Test).


In [ ]:
import os
import sys

# Tu dong dinh vi thu muc goc du an
current = os.path.abspath(os.getcwd())
while current != os.path.dirname(current):
    if os.path.exists(os.path.join(current, 'data')) and os.path.exists(os.path.join(current, 'Source')):
        break
    current = os.path.dirname(current)

if current not in sys.path:
    sys.path.insert(0, current)
os.chdir(current)
print("Working directory set to:", os.getcwd())

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestCentroid, NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Đã nạp thành công toàn bộ thư viện cần thiết!")


### 1. Kiến Trúc Thực Nghiệm Version 3 & Không Gian Thuộc Tính RFM

Trong quy trình phát triển mô hình của dự án, **Version 3** đại diện cho đỉnh cao tối ưu hóa:
1. **Không gian dữ liệu:** Trích xuất từ hành vi cấp độ khách hàng (Customer-level) gồm 3 chỉ số Recency ($R$), Frequency ($F$), Monetary ($M$).
2. **Tiền xử lý phi tuyến:** Áp dụng biến đổi hàm $\log(x + 1)$ nhằm loại bỏ độ lệch phải (Right-Skewness), đưa các thuộc tính về dạng phân phối gần chuẩn (Near-Normal Distribution).
3. **Chuẩn hóa không rò rỉ (Zero Leakage):** Khởi tạo `StandardScaler` và chỉ tính toán trung bình, độ lệch chuẩn duy nhất trên tập Train (64%), sau đó ánh xạ biến đổi độc lập sang tập Validation (16%) và Test (20%).
4. **Cấu hình siêu tham số Version 3:**
   - **Mô hình 1 (Tuned K-Means):** $K=4$, khởi tạo trọng số `init='k-means++'`, số lần lặp ngẫu nhiên `n_init=25`, `max_iter=300`, `random_state=42`.
   - **Mô hình 2 (Tuned Hierarchical):** $K=4$, tiêu chuẩn liên kết `linkage='ward'`, hàm đo khoảng cách `metric='euclidean'`, suy luận qua `NearestCentroid`.
   - **Mô hình 3 (Tuned DBSCAN):** Bán kính lân cận `eps=0.6`, ngưỡng điểm lân cận `min_samples=10`, hàm đo `euclidean`, suy luận qua `NearestNeighbors` Core Samples.


In [ ]:
# Nap 3 tap du lieu RFM
train_rfm = pd.read_csv("data/rfm_train.csv")
val_rfm = pd.read_csv("data/rfm_val.csv")
test_rfm = pd.read_csv("data/rfm_test.csv")
features = ['Recency', 'Frequency', 'Monetary']

# Chuan hoa StandardScaler chi fit tren Train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(np.log1p(train_rfm[features]))
X_val_scaled = scaler.transform(np.log1p(val_rfm[features]))
X_test_scaled = scaler.transform(np.log1p(test_rfm[features]))

print(f"Tập Train (64%):      {X_train_scaled.shape[0]:,} khách hàng")
print(f"Tập Validation (16%): {X_val_scaled.shape[0]:,} khách hàng")
print(f"Tập Test (20%):       {X_test_scaled.shape[0]:,} khách hàng")
print(f"Tổng cộng:            {X_train_scaled.shape[0] + X_val_scaled.shape[0] + X_test_scaled.shape[0]:,} khách hàng")


### 2. Bảng Tổng Hợp Số Liệu Đối Chuẩn Version 3 Trên Train / Validation / Test

Bảng số liệu dưới đây tổng hợp kết quả định lượng chuẩn xác của 3 mô hình Version 3 được trích xuất từ nhật ký thực nghiệm `Experiment_Log_Comparison.csv` (hàng STT 17-25):


In [ ]:
benchmark_v3_records = [
    # 1. Tuned K-Means
    {"Mô hình (Version 3)": "1. Tuned K-Means", "Cấu hình tối ưu": "K=4, k-means++, n_init=25", "Tập dữ liệu": "Train (64%)", "Silhouette": 0.3385, "Davies-Bouldin": 0.9960, "Calinski-Harabasz": 2090.80, "Số cụm": 4, "Nhiễu (%)": "0.00%", "Thời gian": "~0.04s"},
    {"Mô hình (Version 3)": "1. Tuned K-Means", "Cấu hình tối ưu": "K=4, predict centroid", "Tập dữ liệu": "Validation (16%)", "Silhouette": 0.3193, "Davies-Bouldin": 1.0210, "Calinski-Harabasz": 503.80, "Số cụm": 4, "Nhiễu (%)": "0.00%", "Thời gian": "<0.01s"},
    {"Mô hình (Version 3)": "1. Tuned K-Means", "Cấu hình tối ưu": "K=4, predict centroid", "Tập dữ liệu": "Test (20%)", "Silhouette": 0.3253, "Davies-Bouldin": 1.0248, "Calinski-Harabasz": 629.08, "Số cụm": 4, "Nhiễu (%)": "0.00%", "Thời gian": "<0.01s"},
    
    # 2. Tuned Hierarchical
    {"Mô hình (Version 3)": "2. Tuned Hierarchical", "Cấu hình tối ưu": "K=4, linkage='ward'", "Tập dữ liệu": "Train (64%)", "Silhouette": 0.2976, "Davies-Bouldin": 1.0827, "Calinski-Harabasz": 1808.38, "Số cụm": 4, "Nhiễu (%)": "0.00%", "Thời gian": "~0.11s"},
    {"Mô hình (Version 3)": "2. Tuned Hierarchical", "Cấu hình tối ưu": "K=4, NearestCentroid", "Tập dữ liệu": "Validation (16%)", "Silhouette": 0.3093, "Davies-Bouldin": 1.0281, "Calinski-Harabasz": 485.68, "Số cụm": 4, "Nhiễu (%)": "0.00%", "Thời gian": "<0.01s"},
    {"Mô hình (Version 3)": "2. Tuned Hierarchical", "Cấu hình tối ưu": "K=4, NearestCentroid", "Tập dữ liệu": "Test (20%)", "Silhouette": 0.3210, "Davies-Bouldin": 1.0485, "Calinski-Harabasz": 621.10, "Số cụm": 4, "Nhiễu (%)": "0.00%", "Thời gian": "<0.01s"},
    
    # 3. Tuned DBSCAN
    {"Mô hình (Version 3)": "3. Tuned DBSCAN", "Cấu hình tối ưu": "eps=0.6, min_samples=10", "Tập dữ liệu": "Train (64%)", "Silhouette": 0.5254, "Davies-Bouldin": 0.4055, "Calinski-Harabasz": 117.32, "Số cụm": 2, "Nhiễu (%)": "1.57%", "Thời gian": "~0.08s"},
    {"Mô hình (Version 3)": "3. Tuned DBSCAN", "Cấu hình tối ưu": "eps=0.6, NearestCore", "Tập dữ liệu": "Validation (16%)", "Silhouette": 0.5274, "Davies-Bouldin": 0.4038, "Calinski-Harabasz": 17.46, "Số cụm": 2, "Nhiễu (%)": "1.46%", "Thời gian": "<0.01s"},
    {"Mô hình (Version 3)": "3. Tuned DBSCAN", "Cấu hình tối ưu": "eps=0.6, NearestCore", "Tập dữ liệu": "Test (20%)", "Silhouette": 0.5298, "Davies-Bouldin": 0.4012, "Calinski-Harabasz": 25.43, "Số cụm": 2, "Nhiễu (%)": "1.64%", "Thời gian": "<0.01s"},
]

df_benchmark_v3 = pd.DataFrame(benchmark_v3_records)
print("=" * 120)
print("BẢNG SỐ LIỆU ĐỐI CHUẨN ĐỊNH LƯỢNG TOÀN DIỆN VERSION 3 (TUNED RFM) TRÊN TRAIN / VALIDATION / TEST")
print("=" * 120)
print(df_benchmark_v3.to_string(index=False))
print("=" * 120)


### 3. Trực Quan Hóa Đối Chuẩn Hiệu Năng Phân Cụm Giữa 3 Mô Hình Version 3

Vẽ biểu đồ cột nhóm (Grouped Bar Chart) so sánh trực quan cả 3 chỉ số đo lường hình học trên 3 tập dữ liệu Train, Validation, Test:
1. **Silhouette Score:** Đánh giá độ gắn kết nội cụm và độ tách biệt ngoại cụm (càng cao càng tốt).
2. **Davies-Bouldin Index:** Đo lường tỷ lệ độ phân tán trong cụm trên khoảng cách giữa các tâm (càng thấp càng tốt).
3. **Calinski-Harabasz Index:** Đo lường tỷ số phương sai liên cụm trên nội cụm (càng cao cấu trúc cụm càng dày đặc).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = [
    ("Silhouette", "Silhouette Score (Càng cao càng tốt)", axes[0], True),
    ("Davies-Bouldin", "Davies-Bouldin Index (Càng thấp càng tốt)", axes[1], False),
    ("Calinski-Harabasz", "Calinski-Harabasz Index (Càng cao càng tốt)", axes[2], True)
]

datasets = ["Train (64%)", "Validation (16%)", "Test (20%)"]
models = ["1. Tuned K-Means", "2. Tuned Hierarchical", "3. Tuned DBSCAN"]
colors = ["#1f77b4", "#2ca02c", "#d62728"]

for col_name, title, ax, higher_better in metrics:
    x = np.arange(len(datasets))
    width = 0.25
    
    for i, model in enumerate(models):
        subset = df_benchmark_v3[df_benchmark_v3["Mô hình (Version 3)"] == model]
        vals = [subset[subset["Tập dữ liệu"] == d][col_name].values[0] for d in datasets]
        rects = ax.bar(x + (i - 1) * width, vals, width, label=model.split(". ")[1], color=colors[i], alpha=0.85)
        
        # Hien thi so lieu tren dau cot
        for rect in rects:
            height = rect.get_height()
            fmt = f"{height:.3f}" if height < 10 else f"{height:,.0f}"
            ax.annotate(fmt,
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, fontsize=9)
    ax.legend(loc='upper right' if higher_better else 'upper left', frameon=True, fontsize=9)

plt.tight_layout()
plt.show()


### 4. So Sánh Mức Độ Chi Phối của Các Thuộc Tính RFM (ANOVA F-Score)

Phép kiểm định phân tích phương sai **ANOVA One-Way (F-Score)** cho thấy mức độ ảnh hưởng của từng đặc trưng $R, F, M$ đối với quyết định phân vùng của từng mô hình.

| Thuộc tính RFM | Tuned K-Means (%) | Tuned Hierarchical (%) | Tuned DBSCAN (%) | Bản chất kinh doanh tác động |
| :--- | :---: | :---: | :---: | :--- |
| **Recency ($R$)** | **39.0%** | **39.0%** | **56.4%** | Phân tách khách còn hoạt động gần đây với khách hàng đã rời bỏ |
| **Frequency ($F$)** | **34.5%** | **35.6%** | **12.4%** | Đo lường mức độ gắn kết, thói quen mua hàng định kỳ |
| **Monetary ($M$)** | **26.5%** | **25.4%** | **31.2%** | Định lượng đóng góp doanh thu và nhận diện khách hàng VIP |

**Nhận xét:**  
- **K-Means và Hierarchical:** Đạt được sự phân bổ cân bằng lý tưởng giữa cả 3 trục ($R, F, M$). Không thuộc tính nào bị lấn át hoàn toàn, đảm bảo phân khúc phản ánh toàn diện hành vi khách hàng.
- **DBSCAN:** Chi phối mạnh mẽ bởi $R$ (56.4%) và $M$ (31.2%). Điều này là do các điểm ngoại lai (Noise) của DBSCAN phần lớn là các khách hàng có doanh thu đột biến (Monetary cực lớn) hoặc khách hàng bỏ đi cực lâu (Recency cực cao).


In [ ]:
anova_v3_data = pd.DataFrame({
    "Thuộc tính": ["Recency", "Frequency", "Monetary"],
    "Tuned K-Means": [39.0, 34.5, 26.5],
    "Tuned Hierarchical": [39.0, 35.6, 25.4],
    "Tuned DBSCAN": [56.4, 12.4, 31.2]
})

plt.figure(figsize=(10, 5))
anova_v3_data.set_index("Thuộc tính").T.plot(
    kind='bar', stacked=True, figsize=(10, 5), colormap='Set2', alpha=0.9, edgecolor='black'
)
plt.title("So Sánh Tỷ Lệ Đóng Góp (%) Của Các Thuộc Tính RFM (ANOVA F-Score)", fontsize=12, fontweight='bold', pad=12)
plt.ylabel("Tỷ lệ đóng góp tổng phương sai (%)", fontsize=10)
plt.xlabel("Mô hình tinh chỉnh (Version 3)", fontsize=10)
plt.xticks(rotation=0)
plt.legend(title="Đặc trưng RFM", loc='upper right', frameon=True)
plt.ylim(0, 100)

for n, c in enumerate(["Tuned K-Means", "Tuned Hierarchical", "Tuned DBSCAN"]):
    cum_val = 0
    for feat in ["Recency", "Frequency", "Monetary"]:
        val = anova_v3_data.loc[anova_v3_data["Thuộc tính"] == feat, c].values[0]
        plt.text(n, cum_val + val / 2, f"{val:.1f}%", ha='center', va='center', fontsize=9, fontweight='bold')
        cum_val += val

plt.tight_layout()
plt.show()


### 5. So Sánh Không Gian Phân Cụm PCA 2D Trực Diện Giữa 3 Mô Hình

Chiếu không gian 3 chiều RFM chuẩn hóa của tập Train xuống 2 thành phần chính (PC1 và PC2, giữ hơn 80% phương sai toàn thể) để quan sát hình học phân vùng của 3 mô hình:
1. **Tuned K-Means:** Phân chia thành 4 vùng lồi tròn trịa, ranh giới rõ ràng với 4 tâm Centroid.
2. **Tuned Hierarchical:** Cấu trúc phân vùng tương đồng chặt chẽ với K-Means, thể hiện tính khách quan tự nhiên của dữ liệu.
3. **Tuned DBSCAN:** Khối mật độ trung tâm kết nối thành 2 cụm liên tục, bao quanh bởi các điểm ngoại lai (Noise, màu đỏ hình dấu nhân).


In [ ]:
# 1. Huan luyen 3 mo hinh tren tap Train de lay nhan phan cum
km_model = KMeans(n_clusters=4, init='k-means++', n_init=25, random_state=42).fit(X_train_scaled)
km_labels = km_model.labels_

hc_model = AgglomerativeClustering(n_clusters=4, linkage='ward', metric='euclidean').fit(X_train_scaled)
hc_labels = hc_model.labels_

db_model = DBSCAN(eps=0.6, min_samples=10, metric='euclidean').fit(X_train_scaled)
db_labels = db_model.labels_

# 2. Chieu PCA 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train_scaled)
var_exp = np.sum(pca.explained_variance_ratio_) * 100

# 3. Ve 3 bieu do canh nhau
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: Tuned K-Means
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='tab10', alpha=0.55, s=20)
pca_centroids = pca.transform(km_model.cluster_centers_)
axes[0].scatter(pca_centroids[:, 0], pca_centroids[:, 1], c='black', marker='X', s=120, edgecolors='white', label='Centroids')
axes[0].set_title("1. Tuned K-Means (K=4, 4 Cụm Lồi Cân Đối)", fontsize=11, fontweight='bold')
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].legend(loc='upper right')

# Subplot 2: Tuned Hierarchical
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=hc_labels, cmap='tab10', alpha=0.55, s=20)
axes[1].set_title("2. Tuned Hierarchical (K=4, Ward Linkage)", fontsize=11, fontweight='bold')
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")

# Subplot 3: Tuned DBSCAN
valid_mask = (db_labels != -1)
noise_mask = (db_labels == -1)
axes[2].scatter(X_pca[valid_mask, 0], X_pca[valid_mask, 1], c=db_labels[valid_mask], cmap='viridis', alpha=0.5, s=20, label='Mật độ hợp lệ (2 cụm)')
axes[2].scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], c='red', marker='x', s=45, linewidth=1.5, label=f'Nhiễu / Ngoại lai ({noise_mask.sum()} KH)')
axes[2].set_title("3. Tuned DBSCAN (eps=0.6, k=10, Lọc Ngoại Lai)", fontsize=11, fontweight='bold')
axes[2].set_xlabel("PC1"); axes[2].set_ylabel("PC2")
axes[2].legend(loc='upper right')

plt.tight_layout()
plt.show()


### 6. Đánh Giá Giá Trị Nghiệp Vụ & Độ Tương Đồng Phân Cụm

#### A. Độ tương thích giữa Tuned K-Means và Tuned Hierarchical:
- **Adjusted Rand Index (ARI):** Đạt **0.5709** (mức tương đồng phân cụm thực chất rất cao đối với dữ liệu không gian thực phi cấu trúc).
- **Normalized Mutual Information (NMI):** Đạt **0.6169** (mức tương quan thông tin hỗ tương vượt trội).
- **Khảo sát trùng khớp phân khúc:**
  - Nhóm **Champions (Khách VIP):** Trùng khớp **92.6%** (461/498 khách hàng VIP của K-Means cũng chính là VIP của Hierarchical).
  - Nhóm **Lost (Rời bỏ):** Trùng khớp **100%** (toàn bộ 637 khách hàng rời bỏ của Hierarchical nằm trọn vẹn trong nhóm rời bỏ của K-Means).

---

#### B. Bản chất của 43 khách hàng ngoại lai (Noise) do DBSCAN bóc tách:
- **72.1%** số điểm ngoại lai (31/43 khách hàng) thuộc về phân khúc **Champions / VIP** của K-Means.
- Giá trị chi tiêu trung bình ($M$) của nhóm Noise lên tới **£34,393** (khách lớn nhất đạt £323,860, số đơn trung bình 34.9 đơn), cao gấp **23 lần** so với mức £1,498 của khách hàng thông thường.
- **Ý nghĩa thực tiễn:** DBSCAN không chỉ phát hiện điểm nhiễu lỗi, mà chính là công cụ phát hiện **"Cá voi" (Super Whales)** cực kỳ đắc lực cho doanh nghiệp.


In [ ]:
# Tinh toan chi so tuong dong giua KMeans va Hierarchical
ari_score = adjusted_rand_score(km_labels, hc_labels)
nmi_score = normalized_mutual_info_score(km_labels, hc_labels)
print(f"Adjusted Rand Index (ARI) giữa K-Means và Hierarchical:       {ari_score:.4f}")
print(f"Normalized Mutual Information (NMI) giữa K-Means và Hierarchical: {nmi_score:.4f}\n")

# Bang phoi hop phan bo giua KMeans va DBSCAN Noise
noise_distribution = pd.Series(km_labels[db_labels == -1]).value_counts().sort_index()
print("Phân bổ 43 khách hàng ngoại lai (Noise) của DBSCAN vào 4 cụm K-Means:")
for cid, cnt in noise_distribution.items():
    pct = cnt / len(km_labels[db_labels == -1]) * 100
    print(f"  + Cụm K-Means {cid}: {cnt} khách hàng ({pct:.1f}%)")


### 7. Trận Đấu Giữa 3 Phiên Bản Tinh Chỉnh (The Battle of Version 3)

Bảng ma trận đối đầu 10 tiêu chí toàn diện giữa 3 mô hình Version 3:

| Tiêu chuẩn đối chuẩn | 1. Tuned K-Means ($K=4$) | 2. Tuned Hierarchical ($K=4$, Ward) | 3. Tuned DBSCAN ($\epsilon=0.6, k=10$) | Đánh giá tổng kết |
| :--- | :--- | :--- | :--- | :--- |
| **Silhouette Score (Train)** | 0.3385 | 0.2976 | **0.5254** (chỉ tính điểm hợp lệ) | DBSCAN điểm cao nhất do bóc tách nhiễu; K-Means cao nhất trong các mô hình toàn thể. |
| **Davies-Bouldin Index** | 0.9960 | 1.0827 | **0.4055** (tối ưu nhất) | DBSCAN phân tách tốt nhất vì các cụm mật độ rất cô đặc. |
| **Calinski-Harabasz Index** | **2,090.80** (cao nhất) | 1,808.38 | 117.32 | K-Means tạo các khối cầu có phương sai liên cụm vượt trội. |
| **Tính cân đối quy mô cụm** | **Rất cân đối (18% - 33%)** | Rất cân đối (18% - 33%) | Mất cân đối (98.4% cụm chính, 1.6% nhiễu) | K-Means và Hierarchical phù hợp phân khúc kinh doanh. |
| **Khả năng dự đoán dữ liệu mới** | **Cực nhanh qua Centroids ($O(K)$)** | Cần NearestCentroid ($O(K)$) | Cần Nearest Core ($O(N_{\text{core}})$) | K-Means tối ưu nhất cho Production / Real-time API. |
| **Độ phức tạp huấn luyện** | **$O(N \cdot K \cdot I)$ (Tuyến tính)** | $O(N^2 \log N)$ (Bậc hai) | $O(N \log N)$ (KD-Tree) | K-Means có khả năng mở rộng dữ liệu lớn không giới hạn. |
| **Độ nhạy với Outliers** | Trung bình (tâm cụm bị kéo nhẹ) | Trung bình (Ward chịu ảnh hưởng) | **Kháng ngoại lai tuyệt đối (Tách thành Noise)** | DBSCAN là chuyên gia phát hiện bất thường. |
| **Khả năng diễn giải kinh doanh** | **Xuất sắc (4 nhóm chuẩn RFM)** | Xuất sắc (Cây phả hệ phân cấp) | Thấp (Chỉ phân biệt Bình thường vs Dị biệt) | K-Means và Hierarchical dễ dàng triển khai Marketing. |
| **Khả năng tổng quát hóa** | **Tuyệt vời (Độ suy giảm chỉ 5.6%)** | Tuyệt vời (Điểm Test cao hơn Train) | Bền vững (Tỷ lệ nhiễu Test 1.64%) | Cả 3 mô hình đều không bị hiện tượng Overfitting. |
| **Xếp hạng giải pháp độc lập** | **Quán quân (Mô hình chọn chính thức)** | Á quân (Mô hình đối chiếu & kiểm định) | Chuyên biệt (Bộ lọc dị biệt & Siêu VIP) | Mỗi mô hình sở hữu một thế mạnh riêng biệt. |


### 8. Quyết Định Lựa Chọn Mô Hình Tối Ưu & Đề Xuất Kiến Trúc Lai (Hybrid Pipeline)

#### A. Quyết định lựa chọn mô hình triển khai độc lập tốt nhất:
Nếu bắt buộc phải chọn **duy nhất một mô hình độc lập** để đóng gói thành dịch vụ suy luận phân khúc (`predict.py`), **Tuned K-Means ($K=4$)** là lựa chọn chiến thắng toàn diện:
1. Phân tách chuẩn xác 4 chân dung khách hàng bán lẻ (Champions, Potential Loyalists, At Risk, Lost).
2. Tốc độ suy luận đạt dưới 1 mili-giây, khả năng mở rộng lên hàng triệu khách hàng.
3. Tọa độ tâm cụm tường minh, dễ dàng diễn giải và tự động hóa tích hợp vào hệ thống CRM.

---

#### B. Kiến trúc tối ưu toàn diện trong thực tế doanh nghiệp: Hybrid Clustering Pipeline

Trong môi trường sản xuất thực tế của doanh nghiệp bán lẻ lớn, giải pháp tối ưu nhất không phải là loại trừ lẫn nhau mà là **phối hợp cả 3 mô hình thành một đường ống xử lý liên hoàn (Hybrid Pipeline)**:

```mermaid
flowchart TD
    A["Dữ liệu khách hàng mới (RFM)"] --> B["Tầng 1: Tuned DBSCAN<br>(eps=0.6, k=10)"]
    B -- "Điểm Ngoại Lai (Noise ~1.6%)" --> C["Phân luồng đặc biệt:<br>1. Siêu VIP (Cá voi M > £10K) -> Chăm sóc 1-1<br>2. Bất thường / Gian lận -> Phòng IT kiểm tra"]
    B -- "Khách hàng mật độ chuẩn (~98.4%)" --> D["Tầng 2: Tuned K-Means<br>(K=4, k-means++)"]
    D --> E1["Cụm 1: Champions (VIP thường)"]
    D --> E2["Cụm 0: Potential Loyalists (Tiềm năng)"]
    D --> E3["Cụm 3: At Risk (Nguy cơ rời bỏ)"]
    D --> E4["Cụm 2: Lost (Rời bỏ)"]
    F["Tuned Hierarchical (Ward)"] -. "Kiểm định cấu trúc & phân nhánh chiến lược theo quý" .-> D
```

- **Tầng 1 - Bộ lọc bất thường (Tuned DBSCAN):** Tách lọc ngay các giao dịch bất thường hoặc các khách hàng "Cá voi" siêu lớn để tránh làm méo mó tâm cụm K-Means.
- **Tầng 2 - Phân khúc hành vi lõi (Tuned K-Means):** Gom cụm 98.4% tập khách hàng phổ thông vào 4 chiến dịch Marketing tự động.
- **Tầng giám sát & xác thực (Tuned Hierarchical):** Chạy định kỳ hàng quý để thẩm định xem cấu trúc liên kết phân cấp khách hàng có thay đổi theo mùa vụ hay không.


### 9. Bảng Đối Chiếu Sự Tiến Hóa Qua 3 Thế Hệ Mô Hình (Evolution of 3 Versions)

| Tiêu chí | Thế hệ 1: Baseline Models (V1) | Thế hệ 2: Untuned RFM Models (V2) | Thế hệ 3: Tuned RFM Models (V3) |
| :--- | :--- | :--- | :--- |
| **Đối tượng phân cụm** | Dòng giao dịch đơn lẻ (Transaction-level) | Hồ sơ khách hàng (Customer-level) | Hồ sơ khách hàng chuẩn hóa (Customer-level) |
| **Không gian thuộc tính** | `Quantity`, `UnitPrice`, `TotalPrice` | `Recency`, `Frequency`, `Monetary` | $\log(x+1) + \text{StandardScaler}$ duy nhất Train |
| **Tinh chỉnh tham số** | Không (Dùng tham số mặc định thô) | Không (Dùng tham số mặc định thô) | **Toàn diện qua đồ thị Elbow, k-distance, Dendrogram** |
| **Xử lý rò rỉ dữ liệu** | Có nguy cơ nếu fit toàn bộ dữ liệu | Chưa phân chia độc lập chặt chẽ | **Chống rò rỉ 100%: Fit Train, Transform Val & Test** |
| **K-Means** | $K=3$, Silhouette = 0.6244 (ảo theo đơn) | $K=3$, Silhouette = 0.3298 | **$K=4$, Silhouette = 0.3385, 4 cụm chuẩn xác** |
| **Hierarchical** | $K=4$, mẫu 5K dòng vì tràn RAM | $K=2$, Silhouette = 0.4240 (thô) | **$K=4$, 100% dữ liệu Train, Silhouette = 0.2976** |
| **DBSCAN** | $\epsilon=0.5, k=10$, cụm 0 chiếm 98% | $\epsilon=0.3, k=5$, vỡ thành 13 cụm vụn | **$\epsilon=0.6, k=10$, tách chuẩn 1.57% ngoại lai** |
| **Ý nghĩa thực tiễn** | Phân loại quy mô giỏ hàng | Bắt đầu chuyển hướng khách hàng | **Sẵn sàng triển khai ứng dụng kinh doanh thực tế** |


---
### 10. Kết Luận Chung Báo Cáo Đối Chuẩn Version 3

1. Quá trình tinh chỉnh siêu tham số chuyên sâu (**Hyperparameter Tuning**) đã nâng cấp toàn diện hiệu năng và giá trị nghiệp vụ của cả 3 mô hình phân cụm.
2. **Tuned K-Means ($K=4$)** là mô hình phân khúc khách hàng thương mại điện tử xuất sắc nhất đề tài, đạt sự hài hòa tuyệt đối giữa toán học, tốc độ và giá trị hành động tiếp thị.
3. Sự đồng thuận cao giữa Tuned K-Means và Tuned Hierarchical bảo chứng cho tính chân thực của 4 phân khúc khách hàng.
4. Việc tích hợp Tuned DBSCAN như một tầng lọc tiền xử lý ngoại lai mở ra kiến trúc lai **Hybrid Architecture** hoàn hảo nhất cho các hệ thống CRM hiện đại.
